# Intermediate Python — Iterators & Generators
### 25 Questions: the `for` loop protocol, `iter`/`next`, custom iterators, `yield`, generator expressions

**Why this matters for pandas:** things like `df.iterrows()`, `df.itertuples()`, and chunked CSV reading (`pd.read_csv(..., chunksize=1000)`) are all generators under the hood — they hand you one row/chunk at a time instead of loading everything into memory at once. Understanding this by hand explains WHY pandas can handle files too big to fit in memory.

**The core idea:** every `for` loop you've ever written has secretly been calling `iter()` then `next()` repeatedly behind the scenes. This notebook makes that visible.

Same rules: attempt first, run your own cell, then compare.

---

## Part A: What a `for` Loop Actually Does (Q1-Q6)

**Q1** Given `scores = [720, 610, 800]`, call `iter(scores)` to get an ITERATOR object, storing it in `it`. Print `it` and `type(it)` — notice it's a different type than the list itself (a `list_iterator`).

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
scores = [720, 610, 800]
it = iter(scores)
print(it)
print(type(it))

**Q2** Using the `it` iterator from Q1, call `next(it)` THREE times in a row (three separate calls, not a loop), printing each result — observe it gives you one value at a time, advancing each call.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
scores = [720, 610, 800]
it = iter(scores)
print(next(it))
print(next(it))
print(next(it))

**Q3** Call `next(it)` a FOURTH time on the now-exhausted iterator from Q2 — confirm it raises `StopIteration`. Catch it with try/except and print `"No more items"`.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
scores = [720, 610, 800]
it = iter(scores)
next(it)
next(it)
next(it)
try:
    next(it)
except StopIteration:
    print("No more items")

**Q4** Manually reproduce what a `for` loop does under the hood: given `scores = [720, 610, 800]`, use `iter()` and a `while True:` loop calling `next()`, catching `StopIteration` with `break`, printing each value. This is EXACTLY what `for s in scores:` does automatically.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
scores = [720, 610, 800]
it = iter(scores)
while True:
    try:
        value = next(it)
        print(value)
    except StopIteration:
        break

**Q5** Call `iter()` TWICE on the same list, storing two SEPARATE iterator objects `it1` and `it2`. Advance `it1` by calling `next()` once, then confirm `it2` is UNAFFECTED (still starts from the beginning) by calling `next(it2)` and seeing the first element.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
scores = [720, 610, 800]
it1 = iter(scores)
it2 = iter(scores)

print(next(it1))
print(next(it2))
# it2 is a completely independent iterator — advancing it1 didn't touch it

**Q6** Try calling `next()` directly on a LIST (not an iterator) — e.g. `next(scores)` — confirm this raises `TypeError`, proving a list itself is ITERABLE (can produce an iterator via `iter()`) but is NOT an iterator itself (has no `next()`).

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
scores = [720, 610, 800]
try:
    next(scores)
except TypeError as e:
    print("Error:", e)
# Lists are iterable (support iter()) but are not themselves iterators.

## Part B: Building a Custom Iterator Class (Q7-Q13)

**Q7** Build a class `ScoreIterator` implementing the iterator protocol manually: `__init__(self, scores)` stores the list and an index `self.index = 0`; `__iter__(self)` returns `self`; `__next__(self)` returns the next score and increments the index, raising `StopIteration` when the index runs out. Create one from `[720, 610, 800]` and loop through it with a `for` loop to confirm it works like any built-in iterable.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
class ScoreIterator:
    def __init__(self, scores):
        self.scores = scores
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.scores):
            raise StopIteration
        value = self.scores[self.index]
        self.index += 1
        return value

for s in ScoreIterator([720, 610, 800]):
    print(s)

**Q8** Using the `ScoreIterator` class from Q7, also test it MANUALLY with `iter()`/`next()` instead of a `for` loop — since `__iter__` returns `self`, confirm `iter(si) is si` is `True` (the object IS its own iterator).

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
class ScoreIterator:
    def __init__(self, scores):
        self.scores = scores
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.scores):
            raise StopIteration
        value = self.scores[self.index]
        self.index += 1
        return value

si = ScoreIterator([720, 610, 800])
print(iter(si) is si)
print(next(si))
print(next(si))

**Q9** Build a custom iterator class `RiskFilterIterator` that wraps a list of scores but SKIPS any score below 650 automatically inside `__next__` (loop internally until a qualifying value is found or the list is exhausted). Test it on `[720, 610, 800, 590, 700]` with a `for` loop — it should only yield 720, 800, 700.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
class RiskFilterIterator:
    def __init__(self, scores):
        self.scores = scores
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self.index < len(self.scores):
            value = self.scores[self.index]
            self.index += 1
            if value >= 650:
                return value
        raise StopIteration

for s in RiskFilterIterator([720, 610, 800, 590, 700]):
    print(s)

**Q10** Build a class `Countdown` that is ITERABLE (has `__iter__` returning a NEW separate iterator object each time, unlike Q7-9 where the object was its own iterator) — implement it using a nested helper or by returning `iter(range(self.start, 0, -1))` directly from `__iter__`. Test with `Countdown(5)` in a `for` loop, printing the countdown.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return iter(range(self.start, 0, -1))

for n in Countdown(5):
    print(n)

**Q11** Confirm the KEY BENEFIT of Q10's design: loop through the SAME `Countdown(3)` object TWICE (two separate `for` loops on the same instance) — both should print `3, 2, 1` fully, because each `for` loop calls `__iter__` fresh, getting a NEW iterator each time. (Contrast this with Q7's `ScoreIterator`, which would be EXHAUSTED after one loop since it returns `self`.)

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return iter(range(self.start, 0, -1))

c = Countdown(3)
for n in c:
    print(n)
print("---")
for n in c:
    print(n)
# Works fully both times — a fresh iterator is created each time __iter__ runs.

**Q12** Now demonstrate the OPPOSITE with `ScoreIterator` from Q7: create ONE instance, loop through it with a `for` loop ONCE (prints all values), then loop through the SAME instance AGAIN — confirm the second loop prints NOTHING, because the iterator was already exhausted and `__iter__` just returns `self` (already at the end).

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
class ScoreIterator:
    def __init__(self, scores):
        self.scores = scores
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.scores):
            raise StopIteration
        value = self.scores[self.index]
        self.index += 1
        return value

si = ScoreIterator([720, 610, 800])
for s in si:
    print(s)
print("---")
for s in si:
    print(s)
print("(second loop printed nothing — iterator was exhausted)")

**Q13** Explain in a comment: based on Q10-12, when designing your own class, would you rather make it "an iterable that PRODUCES iterators" (like `Countdown`) or "an iterator that IS itself" (like `ScoreIterator`)? Which is more reusable, and why?

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
# "An iterable that produces fresh iterators" (Countdown-style) is almost
# always more reusable — you can loop over the same object multiple times,
# use it in nested loops, pass it to multiple functions that each loop over
# it independently, etc. without worrying about exhaustion.
#
# "An object that IS its own iterator" (ScoreIterator-style) is simpler to
# write but can only be consumed ONCE — a real gotcha if some other part of
# your code loops over it expecting to still see items after a previous
# loop already consumed them. This is exactly the surprise people hit with
# actual generators too (covered next) — they're single-use by design.
print("See comment above")

## Part C: Generator Functions with `yield` (Q14-Q21)

**Q14** Write a GENERATOR FUNCTION `count_up_to(n)` using `yield` instead of `return` — it should yield `1, 2, 3, ..., n` one at a time. Call it (`gen = count_up_to(5)`), print `gen` and `type(gen)` to see it's a generator object, THEN loop through it with a `for` loop to print all values.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(5)
print(gen)
print(type(gen))

for value in gen:
    print(value)

**Q15** Call `count_up_to(3)` (from Q14) and use `next()` on it MANUALLY three times (not a for loop), printing each result — confirm a generator supports `next()` just like the custom iterator classes did, but you got this for free just by using `yield`.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(3)
print(next(gen))
print(next(gen))
print(next(gen))

**Q16** Call `next()` a FOURTH time on the exhausted generator from Q15 — confirm it raises `StopIteration`, exactly like a custom iterator would.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(3)
next(gen)
next(gen)
next(gen)
try:
    next(gen)
except StopIteration:
    print("Generator exhausted")

**Q17** Rewrite the `RiskFilterIterator` CLASS from Q9 as a GENERATOR FUNCTION `filter_low_risk(scores)` using `yield` inside a simple `for`/`if` loop — notice how much shorter this is than the class-based version. Test on `[720, 610, 800, 590, 700]`.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
def filter_low_risk(scores):
    for s in scores:
        if s >= 650:
            yield s

for s in filter_low_risk([720, 610, 800, 590, 700]):
    print(s)

**Q18** Write a generator function `running_total(numbers)` that yields the CUMULATIVE sum as it goes (e.g. for `[100, 200, 300]`, yields `100, 300, 600`) — track a running total variable inside the function, yielding it after each addition. Test it.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
def running_total(numbers):
    total = 0
    for n in numbers:
        total += n
        yield total

for t in running_total([100, 200, 300]):
    print(t)

**Q19** Write an INFINITE generator `infinite_counter(start=0)` using `while True: yield ...` (never stops on its own). Since you can't loop it fully, use `itertools.islice` (import `islice` from `itertools`) to take just the FIRST 5 values safely: `list(islice(infinite_counter(), 5))`.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
from itertools import islice

def infinite_counter(start=0):
    n = start
    while True:
        yield n
        n += 1

first_five = list(islice(infinite_counter(), 5))
print(first_five)

**Q20** Write a generator function `read_in_batches(items, batch_size)` that yields LISTS of `batch_size` items at a time from a bigger list (the last batch may be smaller). Test with `list(range(1, 11))` and `batch_size=3` — this is EXACTLY the pattern behind `pd.read_csv(..., chunksize=N)`.

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
def read_in_batches(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

for batch in read_in_batches(list(range(1, 11)), 3):
    print(batch)

**Q21** Explain in a comment the MEMORY benefit of a generator over building a full list upfront — specifically, why `read_in_batches` (Q20) as a generator is better for a HUGE dataset than a function that builds and returns the complete list of all batches at once.

In [ ]:
# YOUR CODE HERE


**Solution 21**

In [ ]:
# A generator produces ONE value (or batch) at a time, on demand, and does
# NOT hold the whole result in memory simultaneously — only the current
# position/state needs to be tracked. If you built "all batches" as one big
# list upfront, you'd need enough memory to hold every batch at once, even
# if you only ever process one batch at a time.
#
# For a dataset with millions of rows, this is the difference between
# "runs fine" and "crashes with a memory error" — this is literally why
# pd.read_csv(..., chunksize=N) exists: it returns a generator-like object
# so you can process a huge CSV file that would never fit in memory all at once.
print("See comment above")

## Part D: Generator Expressions (Q22-Q23)

**Q22** Compare a LIST comprehension `[x**2 for x in range(5)]` (builds the whole list immediately) with a GENERATOR EXPRESSION `(x**2 for x in range(5))` (same syntax, but parentheses instead of brackets — produces a generator, lazily). Print `type()` of each to confirm they're different, then convert the generator to a list with `list()` to see its values.

In [ ]:
# YOUR CODE HERE


**Solution 22**

In [ ]:
list_comp = [x**2 for x in range(5)]
gen_exp = (x**2 for x in range(5))

print(type(list_comp))
print(type(gen_exp))
print(list(gen_exp))

**Q23** You've actually been using generator expressions already without naming them — `sum(x for x in range(1000000) if x % 2 == 0)` computes a sum WITHOUT ever building the full list of a million numbers in memory. Run this and print the result, then explain in a comment why this is more memory-efficient than `sum([x for x in range(1000000) if x % 2 == 0])` (with brackets).

In [ ]:
# YOUR CODE HERE


**Solution 23**

In [ ]:
result = sum(x for x in range(1000000) if x % 2 == 0)
print(result)
# With parentheses (generator expression), sum() pulls one value at a time
# and adds it, never holding all million values in memory simultaneously.
# With brackets (list comprehension), Python first builds the ENTIRE list
# of a million numbers, THEN passes it to sum() — using far more memory
# for no benefit, since sum() only needed to see each value once anyway.

## Part E: Mini Pipeline — Generators on Realistic Data (Q24-Q25)

**Q24** Write a generator function `parse_valid_scores(raw_scores)` that takes a list of raw strings like `["720", "abc", "610", "", "800"]`, and YIELDS only the successfully-converted integers, silently skipping ones that fail (use try/except INSIDE the generator). Test it with a `for` loop, and separately, use it directly inside `sum(parse_valid_scores(...))` to get a total without ever materializing an intermediate list.

In [ ]:
# YOUR CODE HERE


**Solution 24**

In [ ]:
def parse_valid_scores(raw_scores):
    for entry in raw_scores:
        try:
            yield int(entry)
        except ValueError:
            continue

raw = ["720", "abc", "610", "", "800"]

for score in parse_valid_scores(raw):
    print(score)

total = sum(parse_valid_scores(raw))
print("Total:", total)

**Q25** The big one: write a generator function `stream_high_risk_accounts(accounts)` that takes a list of account dicts and yields ONLY the ones where `missed_payments >= 2`, one at a time, ALSO adding a computed key `"flagged_at_step"` to each yielded dict showing which position (1-indexed, counting only flagged ones) it was flagged at. Test on:
```python
accounts = [
    {"id": "A1", "missed_payments": 0},
    {"id": "A2", "missed_payments": 3},
    {"id": "A3", "missed_payments": 1},
    {"id": "A4", "missed_payments": 2},
    {"id": "A5", "missed_payments": 4},
]
```
Loop through the generator, printing each yielded dict — confirm only A2, A4, A5 appear, numbered 1, 2, 3.

In [ ]:
# YOUR CODE HERE


**Solution 25**

In [ ]:
def stream_high_risk_accounts(accounts):
    step = 0
    for a in accounts:
        if a["missed_payments"] >= 2:
            step += 1
            a["flagged_at_step"] = step
            yield a

accounts = [
    {"id": "A1", "missed_payments": 0},
    {"id": "A2", "missed_payments": 3},
    {"id": "A3", "missed_payments": 1},
    {"id": "A4", "missed_payments": 2},
    {"id": "A5", "missed_payments": 4},
]

for flagged in stream_high_risk_accounts(accounts):
    print(flagged)

---
## Checkpoint

25 questions: what `for` loops actually do under the hood (`iter`/`next`/`StopIteration`), building custom iterator classes by hand, the crucial difference between "an object that IS its own iterator" (single-use) vs "an iterable that PRODUCES fresh iterators" (reusable), generator functions with `yield` (dramatically shorter than the class-based equivalent), infinite generators with `itertools.islice`, generator expressions vs list comprehensions, and a mini pipeline streaming filtered/flagged records lazily.

**The one thing worth remembering forever:** generators compute values ON DEMAND instead of all at once, which is why they use constant memory regardless of how much data flows through them. That's exactly why `pd.read_csv(path, chunksize=10000)` exists — it lets you process a CSV far bigger than your RAM, one chunk at a time, using precisely this mechanism.

Ready for the statistics notebook, or straight to numpy now?